<a href="https://colab.research.google.com/github/Moses-Bernard/FieldWork_Tracker/blob/main/HRLMultiSeedRun01_03_26.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
from google.colab import drive, files
drive.mount('/content/drive')


Mounted at /content/drive


In [ ]:
!pip install -q git+https://github.com/DLR-RM/stable-baselines3

In [ ]:
!pip install -r "requirements.txt"

In [ ]:
import os
from datetime import datetime

drive_root = "/content/drive/MyDrive/HRL_Experiments/run_20260228_2030"

run_name = datetime.now().strftime("run_%Y%m%d_%H%M")
target_folder = os.path.join(drive_root, "HRL_Experiments", run_name)

os.makedirs(target_folder, exist_ok=True)

print("Saving experiment to:")
print(target_folder)

In [ ]:
!python -c "from src.env.edge_env import EdgeEnv; from src.env.hrl_env import HREdgeEnv; print('Imports OK')"

In [ ]:
!mkdir -p models/ll_nodes models/hrl models/flat results logs

In [ ]:
!python -m src.training.train_low_level_node --num-nodes 5 --timesteps 50000 --seed 0

In [ ]:
!find models/ll_nodes -type f -name "*.zip" | sort

In [ ]:
!python -m src.training.train_full_system --num-nodes 5 --timesteps 200000 --seed 0 --stress-mode normal

In [ ]:
!python -m src.training.train_full_system --num-nodes 5 --timesteps 200000 --seed 0 --stress-mode bursty

In [ ]:
#import os, subprocess, textwrap

os.chdir("/content/drive/MyDrive/HRL_Experiments/run_20260228_2030")
!pip install requirements.txt
#cmd = [
#    "python", "-m", "src.training.train_full_system",
#    "--num-nodes", "5",
#    "--timesteps", "200000",
#    "--seed", "0",
#    "--stress-mode", "normal",
#]

#p = subprocess.run(cmd, capture_output=True, text=True)
#print("RETURN CODE:", p.returncode)

#print("\n===== STDOUT (last 2000 chars) =====\n")
#print(p.stdout[-2000:])

#print("\n===== STDERR (last 4000 chars) =====\n")
#print(p.stderr[-4000:])

In [ ]:
from src.env.edge_env import EdgeEnv
import inspect

print(inspect.signature(EdgeEnv))

In [ ]:
%%writefile src/env/edge_env.py
import random
import numpy as np
import gymnasium
from gymnasium import spaces


class EdgeNode:
    def __init__(self, node_id, cpu_capacity=1.0):
        self.node_id = int(node_id)
        self.cpu_capacity = float(cpu_capacity)
        self.cpu_free = float(cpu_capacity)

        self.memory_free = self.cpu_capacity * 2.0
        self.bandwidth = 1.0

        self.queue_length = 0
        self.queue_work = 0.0

    def process_task(self, task_size):
        waiting = self.queue_work / max(1e-6, self.cpu_capacity)
        service = task_size / max(1e-6, self.cpu_capacity)
        latency = waiting + service

        self.queue_work += task_size
        self.queue_length = int(np.ceil(self.queue_work))

        self.cpu_free = self.cpu_capacity / (1.0 + self.queue_work)

        return float(latency)


class EdgeEnv(gymnasium.Env):

    metadata = {"render_modes": ["human"]}

    def __init__(
        self,
        num_nodes=5,
        max_steps=200,
        seed=None,
        stress_mode="normal",
        stress_test=None,
        switch_step=100,
        switch_to="bursty",
    ):
        super().__init__()

        self.num_nodes = int(num_nodes)
        self.max_steps = int(max_steps)
        self.current_step = 0

        self._seed = seed
        self._rng = random.Random(seed)

        # Stress controls
        self.stress_mode = stress_mode
        self.stress_test = stress_test or {};
        self.switch_step = int(switch_step)
        self.switch_to = switch_to

        self._burst_prob = float(self.stress_test.get("burst_prob", 0.30))
        self._burst_mult_range = tuple(
            self.stress_test.get("burst_mult_range", (2.0, 5.0))
        )

        # Create nodes
        self.nodes = [
            EdgeNode(i, cpu_capacity=self._rng.uniform(1.0, 3.0))
            for i in range(self.num_nodes)
        ]

        self.action_space = spaces.Discrete(self.num_nodes)

        self.observation_space = spaces.Box(
            low=0.0,
            high=1.0,
            shape=(self.num_nodes,),
            dtype=np.float32,
        )

        self.task_size_range = (0.5, 2.0)

    def _active_mode(self):
        if self.stress_mode == "switch":
            if self.current_step < self.switch_step:
                return "normal"
            return self.switch_to
        return self.stress_mode

    def _inject_stress_multiplier(self, active_mode):
        spike_steps = self.stress_test.get("spike_steps")

        if spike_steps and self.current_step in spike_steps:
            return 2.0

        if active_mode == "bursty":
            if self._rng.random() < self._burst_prob:
                lo, hi = self._burst_mult_range
                return self._rng.uniform(lo, hi)

        return 1.0

    def reset(self, seed=None, options=None):
        if seed is None:
            seed = self._seed

        self._rng = random.Random(seed)
        self.current_step = 0

        for node in self.nodes:
            node.cpu_free = node.cpu_capacity
            node.queue_length = 0
            node.queue_work = 0.0

        obs = np.array(
            [n.cpu_free / n.cpu_capacity for n in self.nodes],
            dtype=np.float32,
        )

        return obs, {}

    def step(self, action):
        self.current_step += 1
        done = self.current_step >= self.max_steps

        active_mode = self._active_mode()
        multiplier = self._inject_stress_multiplier(active_mode)

        # Drain queues
        for n in self.nodes:
            n.queue_work = max(0.0, n.queue_work - n.cpu_capacity)
            n.queue_length = int(np.ceil(n.queue_work))
            n.cpu_free = n.cpu_capacity / (1.0 + n.queue_work)

        node_idx = int(action) % self.num_nodes
        node = self.nodes[node_idx]

        task_size = self._rng.uniform(*self.task_size_range) * multiplier
        latency = node.process_task(task_size)
        reward = 1.0 / (latency + 1e-6)

        obs = np.array(
            [n.cpu_free / n.cpu_capacity for n in self.nodes],
            dtype=np.float32,
        )

        phase = None
        if self.stress_mode == "switch":
            phase = "pre" if self.current_step < self.switch_step else "post"

        info = {
            "latency": float(latency),
            "node": int(node_idx),
            "task_size": float(task_size),
            "mode": active_mode,
            "phase": phase,
        }

        return obs, float(reward), done, False, info

    def render(self):
        print(
            f"Step {self.current_step} | CPU free: "
            f"{[round(n.cpu_free,2) for n in self.nodes]}"
        )


In [ ]:
from src.env.edge_env import EdgeEnv

env = EdgeEnv(
    num_nodes=5,
    max_steps=200,
    stress_mode="switch",
    switch_step=100,
    switch_to="bursty",
    seed=0
)

obs, _ = env.reset()

for i in range(105):
    obs, r, done, trunc, info = env.step(0)

print("Step:", env.current_step)
print("Mode:", info["mode"])
print("Phase:", info["phase"])

In [ ]:
from src.env.edge_env import EdgeEnv

env = EdgeEnv(
    num_nodes=5,
    max_steps=200,
    stress_mode="switch",
    switch_step=100,
    switch_to="bursty",
    seed=0
)

obs, _ = env.reset()

for i in range(105):
    obs, r, done, trunc, info = env.step(0)

print("Step:", env.current_step)
print("Mode:", info["mode"])
print("Phase:", info["phase"])

In [ ]:
import os
import subprocess

os.chdir("/content/drive/MyDrive/HRL_Experiments/run_20260228_2030")

SEEDS = [0,1,2,3,4]

for s in SEEDS:
    print(f"==== HRL NORMAL seed={s} ====")
    subprocess.run([
        "python", "-m", "src.training.train_full_system",
        "--num-nodes", "5",
        "--timesteps", "200000",
        "--seed", str(s),
        "--stress-mode", "normal"
    ], check=True)

    print(f"==== HRL BURSTY seed={s} ====")
    subprocess.run([
        "python", "-m", "src.training.train_full_system",
        "--num-nodes", "5",
        "--timesteps", "200000",
        "--seed", str(s),
        "--stress-mode", "bursty"
    ], check=True)

print("==== HRL MODELS CREATED ====")
subprocess.run("find models/hrl -type f -name '*.zip' | sort", shell=True)

==== HRL NORMAL seed=0 ====
==== HRL BURSTY seed=0 ====
==== HRL NORMAL seed=1 ====
==== HRL BURSTY seed=1 ====
==== HRL NORMAL seed=2 ====
==== HRL BURSTY seed=2 ====
==== HRL NORMAL seed=3 ====
==== HRL BURSTY seed=3 ====
==== HRL NORMAL seed=4 ====
==== HRL BURSTY seed=4 ====
==== HRL MODELS CREATED ====


CompletedProcess(args="find models/hrl -type f -name '*.zip' | sort", returncode=0)

In [ ]:
import os
from stable_baselines3 import PPO
from stable_baselines3.common.vec_env import DummyVecEnv
from src.env.edge_env import EdgeEnv

os.makedirs("models/flat", exist_ok=True)

SEEDS = [0,1,2,3,4]
NUM_NODES = 5
TIMESTEPS = 200000

def train_flat(seed, stress_mode, out_path):
    def make_env():
        return EdgeEnv(num_nodes=NUM_NODES, max_steps=200, seed=seed, stress_mode=stress_mode)
    env = DummyVecEnv([make_env])
    model = PPO("MlpPolicy", env, verbose=0, seed=seed)
    model.learn(total_timesteps=TIMESTEPS)
    model.save(out_path)
    print("Saved", out_path)

for s in SEEDS:
    train_flat(s, "normal", f"models/flat/flat_model_seed{s}_normal.zip")
    train_flat(s, "bursty", f"models/flat/flat_model_seed{s}_bursty.zip")

print("==== FLAT MODELS CREATED ====")
import glob
print("\n".join(sorted(glob.glob("models/flat/*.zip"))))

Gym has been unmaintained since 2022 and does not support NumPy 2.0 amongst other critical functionality.
Please upgrade to Gymnasium, the maintained drop-in replacement of Gym, or contact the authors of your software and request that they upgrade.
See the migration guide at https://gymnasium.farama.org/introduction/migration_guide/ for additional information.
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


Saved models/flat/flat_model_seed0_normal.zip
Saved models/flat/flat_model_seed0_bursty.zip
Saved models/flat/flat_model_seed1_normal.zip
Saved models/flat/flat_model_seed1_bursty.zip
Saved models/flat/flat_model_seed2_normal.zip
Saved models/flat/flat_model_seed2_bursty.zip
Saved models/flat/flat_model_seed3_normal.zip
Saved models/flat/flat_model_seed3_bursty.zip
Saved models/flat/flat_model_seed4_normal.zip
Saved models/flat/flat_model_seed4_bursty.zip
==== FLAT MODELS CREATED ====
models/flat/flat_model_seed0_bursty.zip
models/flat/flat_model_seed0_normal.zip
models/flat/flat_model_seed1_bursty.zip
models/flat/flat_model_seed1_normal.zip
models/flat/flat_model_seed2_bursty.zip
models/flat/flat_model_seed2_normal.zip
models/flat/flat_model_seed3_bursty.zip
models/flat/flat_model_seed3_normal.zip
models/flat/flat_model_seed4_bursty.zip
models/flat/flat_model_seed4_normal.zip


In [ ]:
import json, os, numpy as np
from stable_baselines3 import PPO
from src.env.edge_env import EdgeEnv
from src.evaluation.evaluate_hrl import evaluate_hrl

os.makedirs("results", exist_ok=True)

SEEDS = [0,1,2,3,4]
EPISODES = 20
MAX_STEPS = 200
NUM_NODES = 5

def eval_flat(model_path, stress_mode, seed):
    model = PPO.load(model_path)
    env = EdgeEnv(num_nodes=NUM_NODES, max_steps=MAX_STEPS, seed=seed, stress_mode=stress_mode)
    lats = []
    for ep in range(EPISODES):
        obs, _ = env.reset(seed=seed+ep)
        done = False
        while not done:
            a, _ = model.predict(obs, deterministic=True)
            obs, r, done, trunc, info = env.step(int(a))
            lats.append(float(info["latency"]))
            if trunc:
                break
    arr = np.array(lats, dtype=float)
    return {
        "mean_latency": float(arr.mean()),
        "std_latency": float(arr.std()),
        "p95": float(np.percentile(arr, 95)),
        "p99": float(np.percentile(arr, 99)),
        "min": float(arr.min()),
        "max": float(arr.max()),
        "count": int(arr.size),
    }

def eval_hrl(hl_model_path, seed):
    res = evaluate_hrl(
        hl_model_path=hl_model_path,
        ll_nodes_dir="models/ll_nodes",
        episodes=EPISODES,
        max_steps=MAX_STEPS,
        save_path="results/hrl_eval_tmp.json",
        num_nodes=NUM_NODES,
    )
    gs = res["global_summary"]
    return {
        "mean_latency": float(gs["mean_latency"]),
        "std_latency": float(gs["std_latency"]),
        "p95": float(gs["p95"]),
        "p99": float(gs["p99"]),
        "min": float(gs["min"]),
        "max": float(gs["max"]),
        "count": int(gs["count"]),
    }

for s in SEEDS:
    # HRL
    hrl_n_path = f"models/hrl/hl_model_seed{s}.zip"
    hrl_b_path = f"models/hrl/hl_model_seed{s}_bursty.zip"

    hrl_n = eval_hrl(hrl_n_path, s)
    json.dump({"global_summary": hrl_n, "seed": s, "algo": "HRL", "mode": "normal"}, open(f"results/hrl_eval_seed{s}_normal.json","w"), indent=2)

    hrl_b = eval_hrl(hrl_b_path, s)
    json.dump({"global_summary": hrl_b, "seed": s, "algo": "HRL", "mode": "bursty"}, open(f"results/hrl_eval_seed{s}_bursty.json","w"), indent=2)

    # Flat
    flat_n_path = f"models/flat/flat_model_seed{s}_normal.zip"
    flat_b_path = f"models/flat/flat_model_seed{s}_bursty.zip"

    flat_n = eval_flat(flat_n_path, "normal", s)
    json.dump({"global_summary": flat_n, "seed": s, "algo": "Flat", "mode": "normal"}, open(f"results/flat_eval_seed{s}_normal.json","w"), indent=2)

    flat_b = eval_flat(flat_b_path, "bursty", s)
    json.dump({"global_summary": flat_b, "seed": s, "algo": "Flat", "mode": "bursty"}, open(f"results/flat_eval_seed{s}_bursty.json","w"), indent=2)

print("Done. Example files:")
!ls -1 results/*seed0* | head

[HL] Loaded HL model: models/hrl/hl_model_seed0.zip
[LL] Loaded node 0 model: models/ll_nodes/node_0/node_0_ll_policy.zip
[LL] Loaded node 1 model: models/ll_nodes/node_1/node_1_ll_policy.zip
[LL] Loaded node 2 model: models/ll_nodes/node_2/node_2_ll_policy.zip
[LL] Loaded node 3 model: models/ll_nodes/node_3/node_3_ll_policy.zip
[LL] Loaded node 4 model: models/ll_nodes/node_4/node_4_ll_policy.zip
[DEBUG] First info keys seen: ['TimeLimit.truncated', 'allocated', 'latency', 'll_action', 'mode', 'node', 'queue_length', 'queue_work', 'task_size']
[OK] HRL evaluation saved → results/hrl_eval_tmp.json
[DIAG] Missing latency steps: 0
[DIAG] First latency samples: [0.9595303689981012, 1.0784658552987214, 0.44948265557282424, 0.4392353070178411, 0.8346844904687166, 0.4401595175418191, 1.104006389966549, 0.5186826907346528, 0.48288532095792813, 0.6798711715653248]
[HL] Loaded HL model: models/hrl/hl_model_seed0_bursty.zip
[LL] Loaded node 0 model: models/ll_nodes/node_0/node_0_ll_policy.zip
[

In [ ]:
import pandas as pd

print("Counts by algo/mode:")
display(df.groupby(["algo","mode"])["seed"].count().reset_index(name="n"))

print("\nSeeds present per algo/mode:")
display(df.groupby(["algo","mode"])["seed"].apply(lambda x: sorted(x.unique())).reset_index(name="seeds"))

In [ ]:
from scipy.stats import ttest_rel, wilcoxon
import numpy as np
import pandas as pd

def paired_tests(metric, mode):
    # Pivot so we get one row per seed with columns HRL and Flat
    sub = df[df["mode"] == mode][["seed","algo",metric]].copy()
    wide = sub.pivot_table(index="seed", columns="algo", values=metric, aggfunc="mean")

    # Keep only seeds where both exist
    wide = wide.dropna(subset=["HRL","Flat"])

    if len(wide) < 2:
        return {
            "mode": mode, "metric": metric, "n_pairs": int(len(wide)),
            "HRL_mean": float(wide["HRL"].mean()) if len(wide) else None,
            "Flat_mean": float(wide["Flat"].mean()) if len(wide) else None,
            "ttest_p": None, "wilcoxon_p": None
        }

    a = wide["HRL"].to_numpy()
    b = wide["Flat"].to_numpy()

    return {
        "mode": mode,
        "metric": metric,
        "n_pairs": int(len(wide)),
        "HRL_mean": float(a.mean()),
        "Flat_mean": float(b.mean()),
        "ttest_p": float(ttest_rel(a, b).pvalue),
        "wilcoxon_p": float(wilcoxon(a, b).pvalue),
    }

tests = []
for mode in ["normal","bursty"]:
    for metric in ["mean_latency","p95","p99"]:
        tests.append(paired_tests(metric, mode))

tests_df = pd.DataFrame(tests)
display(tests_df)

In [ ]:
from scipy.stats import ttest_rel, wilcoxon
import numpy as np
import pandas as pd

def paired_tests(metric, mode):
    sub = df[df["mode"] == mode][["seed", "algo", metric]].copy()

    # one row per seed, columns = algo
    wide = sub.pivot(index="seed", columns="algo", values=metric)

    # enforce pairing
    wide = wide.dropna(subset=["HRL", "Flat"])

    a = wide["HRL"].to_numpy()
    b = wide["Flat"].to_numpy()

    # paired effect size (Cohen's d on differences)
    diff = a - b
    d = float(diff.mean() / (diff.std(ddof=1) + 1e-12))

    out = {
        "mode": mode,
        "metric": metric,
        "n_pairs": int(len(wide)),
        "HRL_mean": float(a.mean()),
        "Flat_mean": float(b.mean()),
        "delta(HRL-Flat)": float((a - b).mean()),
        "cohens_d_paired": d,
        "ttest_p": float(ttest_rel(a, b).pvalue),
        "wilcoxon_p": float(wilcoxon(a, b).pvalue),
    }
    return out

tests = []
for mode in ["normal", "bursty"]:
    for metric in ["mean_latency", "p95", "p99"]:
        tests.append(paired_tests(metric, mode))

tests_df = pd.DataFrame(tests)
display(tests_df)

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

def seedwise_diff(metric, mode):
    wide = df[df["mode"] == mode].pivot(
        index="seed", columns="algo", values=metric
    ).dropna()
    return wide.index.to_numpy(), (wide["HRL"] - wide["Flat"]).to_numpy()

plt.figure(figsize=(8, 5))

colors = {
    "normal": "#1f77b4",   # blue
    "bursty": "#d62728"    # red
}

for mode in ["normal", "bursty"]:
    seeds, diff = seedwise_diff("p99", mode)

    plt.plot(
        seeds,
        diff,
        marker="o",
        linewidth=2,
        markersize=6,
        label=f"{mode.capitalize()} workload",
        color=colors[mode]
    )

# Zero reference line
plt.axhline(0, linestyle="--", linewidth=1.5, color="black", label="Parity (HRL = Flat)")

plt.xlabel("Seed")
plt.ylabel("Δ p99 latency (HRL − Flat)")
plt.title("Seed-wise Difference in Tail Latency (p99)")
plt.legend()
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

def plot_mode(metric, mode):
    # Pivot to wide format
    wide = df[df["mode"] == mode].pivot(
        index="seed", columns="algo", values=metric
    ).dropna()

    seeds = wide.index.to_numpy()
    hrl_vals = wide["HRL"].to_numpy()
    flat_vals = wide["Flat"].to_numpy()

    plt.figure(figsize=(8, 5))

    # HRL line
    plt.plot(
        seeds,
        hrl_vals,
        marker="o",
        markersize=6,
        linewidth=2.5,
        color="#1f77b4",   # blue
        linestyle="-",
        label="HRL"
    )

    # Flat line
    plt.plot(
        seeds,
        flat_vals,
        marker="s",
        markersize=6,
        linewidth=2.5,
        color="#ff7f0e",   # orange
        linestyle="--",
        label="Flat PPO"
    )

    plt.xlabel("Seed")
    plt.ylabel(metric.replace("_", " ").upper())
    plt.title(f"{metric.upper()} Latency Across Seeds ({mode.capitalize()} Mode)")
    plt.legend()
    plt.grid(alpha=0.3)
    plt.tight_layout()
    plt.show()

# Example:
plot_mode("p99", "bursty")

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

def plot_mode(metric, mode):
    # Pivot to wide format
    wide = df[df["mode"] == mode].pivot(
        index="seed", columns="algo", values=metric
    ).dropna()

    seeds = wide.index.to_numpy()
    hrl_vals = wide["HRL"].to_numpy()
    flat_vals = wide["Flat"].to_numpy()

    plt.figure(figsize=(8, 5))

    # HRL line
    plt.plot(
        seeds,
        hrl_vals,
        marker="o",
        markersize=6,
        linewidth=2.5,
        color="#1f77b4",   # blue
        linestyle="-",
        label="HRL"
    )

    # Flat line
    plt.plot(
        seeds,
        flat_vals,
        marker="s",
        markersize=6,
        linewidth=2.5,
        color="#ff7f0e",   # orange
        linestyle="--",
        label="Flat PPO"
    )

    plt.xlabel("Seed")
    plt.ylabel(metric.replace("_", " ").upper())
    plt.title(f"{metric.upper()} Latency Across Seeds ({mode.capitalize()} Mode)")
    plt.legend()
    plt.grid(alpha=0.3)
    plt.tight_layout()
    plt.show()

# Example:
plot_mode("p99", "normal")

In [ ]:
import numpy as np
import pandas as pd

def cohens_d(a, b):
    a = np.array(a)
    b = np.array(b)
    pooled_std = np.sqrt((a.std()**2 + b.std()**2) / 2)
    return (a.mean() - b.mean()) / pooled_std

results = []

for mode in ["normal", "bursty"]:

    hrl = df[(df.algo=="HRL") & (df.mode==mode)]
    flat = df[(df.algo=="Flat") & (df.mode==mode)]

    for metric in ["mean_latency","p95","p99"]:

        a = hrl.sort_values("seed")[metric].values
        b = flat.sort_values("seed")[metric].values

        d = cohens_d(a,b)

        results.append({
            "mode": mode,
            "metric": metric,
            "HRL_mean": a.mean(),
            "Flat_mean": b.mean(),
            "Cohens_d": d,
            "Percent_change": ((a.mean()-b.mean())/b.mean())*100
        })

effect_df = pd.DataFrame(results)

display(effect_df)

In [ ]:
summary_table = df.groupby(["algo","mode"])[["mean_latency","p95","p99"]].mean().round(3)

display(summary_table)

In [ ]:
import matplotlib.pyplot as plt

normal = df[df.mode=="normal"]

hrl = normal[normal.algo=="HRL"].sort_values("seed")
flat = normal[normal.algo=="Flat"].sort_values("seed")

plt.figure(figsize=(7,4))

plt.plot(hrl.seed, hrl.mean_latency, marker='o', label="HRL", linewidth=2)
plt.plot(flat.seed, flat.mean_latency, marker='s', label="Flat PPO", linewidth=2)

plt.title("Mean Latency Comparison (Normal Workload)")
plt.xlabel("Seed")
plt.ylabel("Mean Latency")
plt.legend()
plt.grid(True)

plt.show()

In [ ]:
bursty = df[df.mode=="bursty"]

hrl = bursty[bursty.algo=="HRL"].sort_values("seed")
flat = bursty[bursty.algo=="Flat"].sort_values("seed")

plt.figure(figsize=(7,4))

plt.plot(hrl.seed, hrl.p99, marker='o', label="HRL", linewidth=2)
plt.plot(flat.seed, flat.p99, marker='s', label="Flat PPO", linewidth=2)

plt.title("Tail Latency (p99) Under Bursty Workloads")
plt.xlabel("Seed")
plt.ylabel("p99 Latency")
plt.legend()
plt.grid(True)

plt.show()

KeyError: False

In [ ]:
import json
import glob

pre = []
post = []

for f in glob.glob("results/*switch*.json"):
    d = json.load(open(f))

    pre.append(d["pre_summary"]["p99"])
    post.append(d["post_summary"]["p99"])

plt.figure(figsize=(6,4))

plt.bar(["Pre-Switch","Post-Switch"], [np.mean(pre),np.mean(post)])

plt.title("Latency Response to Workload Regime Switch")
plt.ylabel("Average p99 Latency")

plt.show()